# Multimodal Generation

The goal: a model that can understand and generate video, audio, and text. This is a key **pre-training focus** across leading video generation labs. The key question: how do you train a single model (or tightly coupled models) to generate coherent multi-modal output? Audio must sync with video. Text descriptions must match visual content.

**~2 hrs, needs GPU for demos.**

## Approaches to Multi-Modal Generation

Four architectural approaches, ordered by coupling strength:

### 1. Separate Models + Post-hoc Alignment (MMAudio)
- Independent models for each modality
- Audio model conditioned on video features after video generation
- MMAudio: 157M params, generates 8s audio in 1.23s
- **Pro:** modular, each model can be optimized independently
- **Con:** alignment is post-hoc, not native; can't truly co-generate

### 2. Dual-Stream DiT (LTX-Video 2)
- Two transformer streams sharing cross-attention
- LTX-2: 14B param video stream + 5B param audio stream
- Streams attend to each other during generation
- **Pro:** models interact during generation, better sync
- **Con:** expensive, complex training, still separate latent spaces

### 3. Unified Latent Space (UniForm)
- Single DiT backbone processes both audio and video tokens
- Audio and video share the same latent space and attention mechanism
- **Pro:** native multi-modal interaction, elegant
- **Con:** latent space must accommodate very different modality structures

### 4. Joint Tokenization (Seedance 2.0)
- Tokenize both audio and video, interleave in a single sequence
- Train a single autoregressive model on the mixed token stream
- Audio tokens and video tokens attend to each other naturally
- **Pro:** simplest architecture, scales naturally
- **Con:** token design is critical, long sequences

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from PIL import Image
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

In [ ]:
# Key insight: audio spectrograms look like images, and diffusion works on them!
# This is how audio diffusion models (Riffusion, AudioLDM) work.

# Generate a synthetic audio signal
sample_rate = 16000
duration = 2.0  # seconds
t = np.linspace(0, duration, int(sample_rate * duration))

# Mix of frequencies (like a chord)
audio = (0.5 * np.sin(2 * np.pi * 440 * t) +   # A4
         0.3 * np.sin(2 * np.pi * 554 * t) +   # C#5
         0.2 * np.sin(2 * np.pi * 659 * t))    # E5

# Add some temporal variation
envelope = np.exp(-t / 0.5) * (1 + 0.3 * np.sin(2 * np.pi * 2 * t))
audio = audio * envelope
audio = audio / np.abs(audio).max()

# Compute mel spectrogram
from scipy.signal import stft
f, t_spec, Zxx = stft(audio, fs=sample_rate, nperseg=512, noverlap=384)
spectrogram = np.abs(Zxx)
log_spec = np.log1p(spectrogram * 100)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Waveform
axes[0].plot(t[:3200], audio[:3200])
axes[0].set_title("Audio Waveform")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")

# Spectrogram (this is the "image" representation)
axes[1].imshow(log_spec, aspect='auto', origin='lower', cmap='magma')
axes[1].set_title("Mel Spectrogram\n(This is the 'image' for audio diffusion)")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Frequency")

# Spectrogram as grayscale image
spec_img = (log_spec - log_spec.min()) / (log_spec.max() - log_spec.min())
axes[2].imshow(spec_img, cmap='gray', aspect='auto', origin='lower')
axes[2].set_title("Spectrogram as Image\n(Input to diffusion model)")
axes[2].set_xlabel("Time")
axes[2].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

# Play the audio
display(Audio(audio, rate=sample_rate))

print("\nKey insight: audio spectrograms ARE images.")
print("Audio diffusion = image diffusion on spectrograms.")
print("To generate audio: noise \u2192 denoise spectrogram \u2192 inverse STFT \u2192 waveform")

In [ ]:
# MMAudio: 157M params, generates synchronized audio from video
# This may or may not fit in 8GB -- try with CPU offloading

try:
    # Try loading MMAudio if available
    # MMAudio uses a custom codebase, so we'll demonstrate the concept
    print("MMAudio architecture:")
    print("  Input: video features (from pre-extracted CLIP/video encoder)")
    print("  + Optional: text prompt for audio style")
    print("  Output: mel spectrogram \u2192 vocoder \u2192 waveform")
    print()
    print("Key design choices:")
    print("  - Flow matching (not DDPM) for audio generation")
    print("  - Multimodal conditioning: video features via cross-attention")
    print("  - 157M params -- small enough to run alongside video model")
    print("  - 1.23s to generate 8s of audio")
    print()
    print("Synchronization mechanism:")
    print("  - Video frames are encoded at regular intervals")
    print("  - Audio spectrogram time axis is aligned to video frame timestamps")
    print("  - Cross-attention between audio tokens and video frame features")
    print("  - The model learns temporal alignment during training")

except Exception as e:
    print(f"Note: {e}")
    print("MMAudio requires custom installation. The key concepts are described above.")

In [ ]:
# Demonstrate how CLIP already creates a shared embedding space for text + images
# This is the foundation for multi-modal generation alignment

from transformers import CLIPModel, CLIPProcessor

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

# Create test data: images + text descriptions
# We'll use synthetic images for self-contained demo
def create_test_image(description: str) -> Image.Image:
    """Create a simple colored image for demo."""
    img = Image.new('RGB', (224, 224))
    pixels = img.load()
    colors = {
        "red": (255, 50, 50), "blue": (50, 50, 255), "green": (50, 200, 50),
        "sunset": (255, 140, 50), "ocean": (30, 100, 200), "forest": (30, 130, 50),
    }
    for key, color in colors.items():
        if key in description.lower():
            for x in range(224):
                for y in range(224):
                    pixels[x, y] = color
            break
    return img

pairs = [
    ("a red flower", "red"),
    ("a blue ocean wave", "ocean"),
    ("a green forest", "forest"),
    ("a sunset over mountains", "sunset"),
]

texts = [p[0] for p in pairs]
images = [create_test_image(p[1]) for p in pairs]

# Encode everything
with torch.no_grad():
    # Text embeddings
    text_inputs = processor(text=texts, return_tensors="pt", padding=True).to(device)
    text_features = model.get_text_features(**text_inputs)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    # Image embeddings
    image_inputs = processor(images=images, return_tensors="pt").to(device)
    image_features = model.get_image_features(**image_inputs)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)

# Compute similarity matrix
similarity = (text_features @ image_features.T).cpu().numpy()

plt.figure(figsize=(8, 6))
plt.imshow(similarity, cmap='RdYlGn', vmin=-0.5, vmax=1.0)
plt.colorbar(label='Cosine Similarity')
plt.xticks(range(len(texts)), [f"img: {p[1]}" for p in pairs], rotation=45)
plt.yticks(range(len(texts)), texts)
plt.title("CLIP: Text-Image Alignment in Shared Embedding Space\n(diagonal = matching pairs)")
plt.tight_layout()
plt.show()

# This is the foundation: align modalities in a shared space
# For multi-modal generation: extend to audio, video, text
print("\nCLIP demonstrates shared embedding space for 2 modalities.")
print("Multi-modal generation extends this to N modalities:")
print("  - ImageBind: images, text, audio, depth, thermal, IMU \u2192 shared space")
print("  - The generation model operates IN this shared space")
print("  - Conditioning across modalities = cross-attention between embeddings")

del model, processor
torch.cuda.empty_cache()

In [ ]:
# Demonstrate the tokenization approach (Seedance 2.0 style)
# Audio and video are tokenized and interleaved in one sequence

# Simulate tokens for a 2-second clip
fps = 8
n_frames = 16
audio_tokens_per_frame = 4  # Audio tokens aligned with each video frame
video_tokens_per_frame = 16  # 4x4 spatial patches per frame

print("Joint Tokenization Example (Seedance 2.0 style)")
print("=" * 60)
print(f"Video: {n_frames} frames, {video_tokens_per_frame} spatial tokens each")
print(f"Audio: {audio_tokens_per_frame} tokens per frame-aligned window")
print(f"Total tokens per frame: {video_tokens_per_frame + audio_tokens_per_frame}")
print(f"Total sequence: {n_frames * (video_tokens_per_frame + audio_tokens_per_frame)} tokens")
print()

# Visualize the token sequence
fig, ax = plt.subplots(figsize=(16, 3))
colors = {'video': '#4CAF50', 'audio': '#FF9800'}
x = 0
for f in range(min(n_frames, 6)):
    # Video tokens for this frame
    for v in range(video_tokens_per_frame):
        ax.barh(0, 1, left=x, color=colors['video'], edgecolor='white', linewidth=0.5)
        x += 1
    # Audio tokens for this frame
    for a in range(audio_tokens_per_frame):
        ax.barh(0, 1, left=x, color=colors['audio'], edgecolor='white', linewidth=0.5)
        x += 1
    # Frame boundary
    ax.axvline(x, color='black', linewidth=1, linestyle='--', alpha=0.3)

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=colors['video'], label='Video tokens'),
                    Patch(color=colors['audio'], label='Audio tokens')],
          loc='upper right')
ax.set_xlabel("Token position in sequence")
ax.set_title("Interleaved Audio-Video Token Sequence\n(AR model processes left to right)")
ax.set_yticks([])
ax.set_xlim(-0.5, x + 0.5)
plt.tight_layout()
plt.show()

print("In AR generation: the model predicts the next token, which could be video OR audio.")
print("The interleaving pattern naturally teaches audio-video synchronization.")
print("Attention across modalities is free -- they're in the same sequence.")

## Recent Production Models: Native Multi-Modal

Gen-4.5 and other recent production models demonstrate multi-modal capabilities:
- Native audio generation alongside video
- Audio editing capabilities
- Multi-shot generation (multiple clips from one prompt, maintaining consistency)

**The pre-training angle:** This requires "re-captioning on the order of billions of samples." Why?
- Multi-modal understanding needs diverse paired data (video + audio + text)
- Existing captions are often text-only, no audio description
- Re-captioning with multi-modal understanding models extracts richer supervision
- Scale matters: billions of samples to cover the distribution of real-world audio-visual events

## Checkpoint

**How does a model learn to sync audio with video?**

### During training:
1. The model sees paired video+audio data (real videos have naturally synchronized audio)
2. Whether using cross-attention, interleaved tokens, or shared latent space -- the model learns temporal correlations between visual and audio events
3. Key: the training data must have TIGHT synchronization. A door closing at frame 47 must have the "slam" sound at the corresponding audio timestamp.

### Architecturally:
- **Cross-attention (MMAudio):** video features at each timestamp attend to audio features and vice versa
- **Interleaved tokens (Seedance):** audio and video tokens at the same timestamp are adjacent in the sequence, so attention naturally connects them
- **Shared latent (UniForm):** both modalities are in the same space, so the model can't help but sync them

### The hard part:
- Not all sounds have visual correspondence (background music, narration)
- Not all visual events have sounds (a butterfly landing)
- Temporal granularity mismatch: audio is 16kHz (16K samples/sec), video is 24fps (24 frames/sec)
- This is why it's a pre-training focus -- you need massive data to learn these subtleties

**Further reading:** MMAudio (2412.15322), LTX-Video 2, Seedance 2.0, UniForm, ImageBind (2305.05665).